In [1]:
print("HELLO")

HELLO


In [3]:
import os
os.chdir("..")  # moves up from notebooks/ to project root
print(os.getcwd())

c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach


In [6]:
from pathlib import Path
print(Path.cwd())
for p in Path.cwd().parents:
    print(p, "->", list(p.glob("data*")))

c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach
c:\Users\Janessa\OneDrive\Desktop\AgenticAi -> []
c:\Users\Janessa\OneDrive\Desktop -> []
c:\Users\Janessa\OneDrive -> []
c:\Users\Janessa -> []
c:\Users -> []
c:\ -> []


In [7]:
"""
Person 3 — Part 1: Train the Audio Confidence Classifier
==========================================================
Trains a Random Forest on data/processed/audio/train.csv to predict
`confidence_label` from extracted audio features (MFCCs, pitch, energy, ZCR).

Run this from the project root (AGENTICAI/AI-Powered-Voic.../).
"""

import pandas as pd
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ---------------------------------------------------------------------------
# 1. Config
# ---------------------------------------------------------------------------
def _find_project_root(marker="data/data/processed/audio/train.csv", max_up=4):
    """Walk up from cwd until we find the expected data file, so this
    works whether run from project root or from notebooks/."""
    p = Path.cwd()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        f"Could not locate '{marker}' by walking up from {Path.cwd()}. "
        "Check that you're inside the project folder."
    )


PROJECT_ROOT = _find_project_root()
TRAIN_PATH = PROJECT_ROOT / "data/data/processed/audio/train.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "confidence_classifier.joblib"
LABEL_ENCODER_PATH = MODEL_DIR / "confidence_label_encoder.joblib"

# Feature columns: 13 MFCCs + pitch/energy/zcr means + duration
FEATURE_COLS = [f"mfcc_{i}" for i in range(13)] + [
    "pitch_mean",
    "energy_mean",
    "zcr_mean",
    "duration",
]
TARGET_COL = "confidence_label"

# ---------------------------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(TRAIN_PATH)
print(f"Loaded train.csv: {df.shape}")
print(f"Label distribution:\n{df[TARGET_COL].value_counts()}\n")

missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected feature columns: {missing}")

X_train = df[FEATURE_COLS]
y_train_raw = df[TARGET_COL]

# ---------------------------------------------------------------------------
# 3. Encode labels (confident / nervous / neutral -> 0/1/2)
# ---------------------------------------------------------------------------
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
print(f"Label classes: {list(label_encoder.classes_)}")

# ---------------------------------------------------------------------------
# 4. Train Random Forest
#    class_weight='balanced' because labels are imbalanced
#    (nervous:538, confident:336, neutral:134 in train.csv)
# ---------------------------------------------------------------------------
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
print(f"\nTrain accuracy (on training set itself, NOT a real metric): {train_acc:.4f}")
print("Real evaluation happens on val.csv/test.csv in Part 2 — not here.")

# ---------------------------------------------------------------------------
# 5. Feature importances (sanity check)
# ---------------------------------------------------------------------------
importances = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(
    ascending=False
)
print("\nTop 5 most important features:")
print(importances.head(5))

# ---------------------------------------------------------------------------
# 6. Save model + label encoder
# ---------------------------------------------------------------------------
joblib.dump(clf, MODEL_PATH)
joblib.dump(label_encoder, LABEL_ENCODER_PATH)
print(f"\nSaved model to: {MODEL_PATH}")
print(f"Saved label encoder to: {LABEL_ENCODER_PATH}")

Loaded train.csv: (1008, 21)
Label distribution:
confidence_label
nervous      538
confident    336
neutral      134
Name: count, dtype: int64

Label classes: ['confident', 'nervous', 'neutral']

Train accuracy (on training set itself, NOT a real metric): 1.0000
Real evaluation happens on val.csv/test.csv in Part 2 — not here.

Top 5 most important features:
duration       0.096733
mfcc_0         0.091332
energy_mean    0.080160
pitch_mean     0.065988
mfcc_2         0.065139
dtype: float64

Saved model to: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\models\confidence_classifier.joblib
Saved label encoder to: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\models\confidence_label_encoder.joblib


In [9]:
"""
Person 3 — Part 2: Evaluate the Audio Confidence Classifier
==============================================================
Loads the model + label encoder saved in Part 1 and evaluates on
data/processed/audio/val.csv and data/processed/audio/test.csv.
Reports accuracy, per-class precision/recall/F1, and confusion matrix.

Run this from the project root or from inside notebooks/ — path is
auto-detected the same way as Part 1.
"""

import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------------------------------------------------------------------------
# 1. Locate project root (same approach as Part 1)
# ---------------------------------------------------------------------------
def _find_project_root(marker="data/data/processed/audio/train.csv", max_up=4):
    p = Path.cwd()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        f"Could not locate '{marker}' by walking up from {Path.cwd()}. "
        "Check that you're inside the project folder."
    )


PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "data/data/processed/audio"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "confidence_classifier.joblib"
LABEL_ENCODER_PATH = MODEL_DIR / "confidence_label_encoder.joblib"

FEATURE_COLS = [f"mfcc_{i}" for i in range(13)] + [
    "pitch_mean",
    "energy_mean",
    "zcr_mean",
    "duration",
]
TARGET_COL = "confidence_label"

# ---------------------------------------------------------------------------
# 2. Load model + label encoder (from Part 1)
# ---------------------------------------------------------------------------
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"No model found at {MODEL_PATH}. Run Part 1 (train_confidence_classifier.py) first."
    )

clf = joblib.load(MODEL_PATH)
label_encoder = joblib.load(LABEL_ENCODER_PATH)
print(f"Loaded model from {MODEL_PATH}")
print(f"Classes: {list(label_encoder.classes_)}\n")


# ---------------------------------------------------------------------------
# 3. Helper: evaluate on one split
# ---------------------------------------------------------------------------
def evaluate_split(csv_path: Path, split_name: str):
    df = pd.read_csv(csv_path)
    missing = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing expected columns: {missing}")

    X = df[FEATURE_COLS]
    y_true_raw = df[TARGET_COL]

    # Guard against unseen labels in val/test that weren't in train
    unseen = set(y_true_raw.unique()) - set(label_encoder.classes_)
    if unseen:
        raise ValueError(
            f"{split_name} contains labels never seen in train.csv: {unseen}"
        )

    y_true = label_encoder.transform(y_true_raw)
    y_pred = clf.predict(X)

    acc = accuracy_score(y_true, y_pred)
    print(f"=== {split_name} ({len(df)} samples) ===")
    print(f"Accuracy: {acc:.4f}\n")
    print("Classification report:")
    print(
        classification_report(
            y_true, y_pred, target_names=label_encoder.classes_, zero_division=0
        )
    )
    print("Confusion matrix (rows=true, cols=predicted):")
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
    print(cm_df)
    print()
    return acc


# ---------------------------------------------------------------------------
# 4. Run on val.csv and test.csv
# ---------------------------------------------------------------------------
val_acc = evaluate_split(DATA_DIR / "val.csv", "VALIDATION")
test_acc = evaluate_split(DATA_DIR / "test.csv", "TEST")

print("=== Summary ===")
print(f"Val accuracy:  {val_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

Loaded model from c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\models\confidence_classifier.joblib
Classes: ['confident', 'nervous', 'neutral']

=== VALIDATION (216 samples) ===
Accuracy: 0.6806

Classification report:
              precision    recall  f1-score   support

   confident       0.63      0.61      0.62        72
     nervous       0.72      0.78      0.75       115
     neutral       0.62      0.45      0.52        29

    accuracy                           0.68       216
   macro avg       0.66      0.61      0.63       216
weighted avg       0.68      0.68      0.68       216

Confusion matrix (rows=true, cols=predicted):
           confident  nervous  neutral
confident         44       24        4
nervous           21       90        4
neutral            5       11       13

=== TEST (216 samples) ===
Accuracy: 0.7222

Classification report:
              precision    recall  f1-score   support

   confident       0.66      0.

In [11]:
import sys
!{sys.executable} -m pip install librosa soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------------------ --------------------- 1.3/2.8 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 7.6 MB/s  0:00:00
   ---------------------------------------- 0.0/43.0 MB ? eta -:--:--
   - -------------------------------------- 2.1/43.0 MB 11.0 MB/s eta 0:00:04
   --- ------------------------------------ 3.4/43.0 MB 8.3 MB/s eta 0:00:05
   ----- ---------------------------------- 5.8/43.0 MB 9.1 MB/s eta 0:00:05
   ------- -------------------------------- 8.1/43.0 MB 9.7 MB/s eta 0:00:04
   ---------- ----------------------------- 10.7/43.0 MB 10.3 MB/s eta 0:00:04
   ------------ --------------------------- 13.1/43.0 MB 10.3 MB/s eta 0:00:03
   -------------- ------------------------- 15.5/43.0 MB 10.5 MB/s eta 0:00:03
   ---------------- ---


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from pathlib import Path

def _find_project_root(marker="data/data/processed/audio/train.csv", max_up=4):
    p = Path.cwd()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f"Could not locate '{marker}' from {Path.cwd()}")

root = _find_project_root()
print("PROJECT_ROOT resolved to:", root)

audio_dir = root / "data/raw/audio_samples"
print("Looking inside:", audio_dir)
print("Does it exist?", audio_dir.exists())

if audio_dir.exists():
    subfolders = sorted([p for p in audio_dir.iterdir() if p.is_dir()])
    print("Subfolders found:", len(subfolders))
    print("First few:", subfolders[:5])

    if subfolders:
        first = subfolders[0]
        print(f"\nContents of {first}:")
        for f in list(first.iterdir())[:10]:
            print(" -", f.name)

PROJECT_ROOT resolved to: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach
Looking inside: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\data\raw\audio_samples
Does it exist? False


In [16]:
from pathlib import Path

def _find_project_root(marker="data/data/processed/audio/train.csv", max_up=4):
    p = Path.cwd()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f"Could not locate '{marker}' from {Path.cwd()}")

root = _find_project_root()
print("PROJECT_ROOT resolved to:", root)

audio_dir = root / "data/raw/audio_samples"
audio_dir_nested = root / "data/data/raw/audio_samples"
print("Checking:", audio_dir, "-> exists?", audio_dir.exists())
print("Checking:", audio_dir_nested, "-> exists?", audio_dir_nested.exists())

# Use whichever one actually exists
if audio_dir_nested.exists():
    audio_dir = audio_dir_nested

print("\nUsing:", audio_dir)

if audio_dir.exists():
    subfolders = sorted([p for p in audio_dir.iterdir() if p.is_dir()])
    print("Subfolders found:", len(subfolders))
    print("First few:", subfolders[:5])

    if subfolders:
        first = subfolders[0]
        print(f"\nContents of {first}:")
        for f in list(first.iterdir())[:10]:
            print(" -", f.name)

PROJECT_ROOT resolved to: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach
Checking: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\data\raw\audio_samples -> exists? False
Checking: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\data\data\raw\audio_samples -> exists? True

Using: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\data\data\raw\audio_samples
Subfolders found: 25
First few: [WindowsPath('c:/Users/Janessa/OneDrive/Desktop/AgenticAi/AI-Powered-Voice-Based-Mock-Interview-Coach/data/data/raw/audio_samples/Actor_01'), WindowsPath('c:/Users/Janessa/OneDrive/Desktop/AgenticAi/AI-Powered-Voice-Based-Mock-Interview-Coach/data/data/raw/audio_samples/Actor_02'), WindowsPath('c:/Users/Janessa/OneDrive/Desktop/AgenticAi/AI-Powered-Voice-Based-Mock-Interview-Coach/data/data/raw/audio_samples/Actor_03'), WindowsPath('c:/Users/Jane

In [18]:
"""
Person 3 — Part 3: Pace / Pause Communication Metrics
========================================================
Workaround for the filler-word limitation documented in the STT report:
instead of counting filler words from the transcript, we measure delivery
quality directly from the audio signal.

- words-per-minute (WPM): transcript word count / audio duration
- pause detection: silence gaps found via librosa energy analysis

NOTE: This has not been run against a real .wav file in this environment
(no internet access to install librosa here) — only checked for logical/
API correctness against librosa's docs. Please run it on a real audio
file and report back if anything errors.
"""

import librosa
import numpy as np


def analyze_pace_and_pauses(
    audio_path: str,
    transcript: str,
    top_db: int = 30,
    min_pause_sec: float = 0.3,
):
    """
    Compute pace (WPM) and pause metrics for a single audio file + its transcript.

    Parameters
    ----------
    audio_path : str
        Path to the .wav file (e.g. data/raw/audio_samples/Actor_01/....wav)
    transcript : str
        The transcript text for this audio (passed in directly — caller decides
        where it comes from: Whisper output, a CSV column, a .txt file, etc.)
    top_db : int
        Threshold (in dB below peak) below which audio is considered silence.
        Lower = stricter (more audio counted as silence). 30 is a reasonable
        default for speech; tune this against a few known samples if pause
        counts look off.
    min_pause_sec : float
        Minimum gap length (seconds) to count as a real pause, filtering out
        tiny sub-phoneme gaps that aren't meaningful pauses.

    Returns
    -------
    dict with:
        duration_sec, word_count, wpm,
        pause_count, total_pause_sec, avg_pause_sec, longest_pause_sec,
        pause_ratio (fraction of total duration spent paused)
    """
    # --- Load audio ---
    y, sr = librosa.load(audio_path, sr=None)
    duration_sec = librosa.get_duration(y=y, sr=sr)

    # --- Pace: words per minute ---
    word_count = len(transcript.split())
    wpm = (word_count / (duration_sec / 60.0)) if duration_sec > 0 else 0.0

    # --- Pause detection via energy-based silence splitting ---
    # librosa.effects.split returns [start, end] sample indices of
    # NON-silent intervals. Gaps BETWEEN consecutive intervals are pauses.
    intervals = librosa.effects.split(y, top_db=top_db)

    pause_durations = []
    if len(intervals) > 1:
        for i in range(len(intervals) - 1):
            gap_start_sample = intervals[i][1]
            gap_end_sample = intervals[i + 1][0]
            gap_sec = (gap_end_sample - gap_start_sample) / sr
            if gap_sec >= min_pause_sec:
                pause_durations.append(gap_sec)

    pause_count = len(pause_durations)
    total_pause_sec = float(sum(pause_durations))
    avg_pause_sec = float(np.mean(pause_durations)) if pause_durations else 0.0
    longest_pause_sec = float(max(pause_durations)) if pause_durations else 0.0
    pause_ratio = (total_pause_sec / duration_sec) if duration_sec > 0 else 0.0

    return {
        "duration_sec": round(duration_sec, 3),
        "word_count": word_count,
        "wpm": round(wpm, 2),
        "pause_count": pause_count,
        "total_pause_sec": round(total_pause_sec, 3),
        "avg_pause_sec": round(avg_pause_sec, 3),
        "longest_pause_sec": round(longest_pause_sec, 3),
        "pause_ratio": round(pause_ratio, 3),
    }


# ---------------------------------------------------------------------------
# Quick manual test (run this file directly to sanity-check on one sample)
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys
    from pathlib import Path

    def _find_project_root(marker="data/data/processed/audio/train.csv", max_up=4):
        p = Path.cwd()
        for _ in range(max_up):
            if (p / marker).exists():
                return p
            p = p.parent
        raise FileNotFoundError(f"Could not locate '{marker}' from {Path.cwd()}")

    root = _find_project_root()
    # Auto-discover any real .wav file under data/raw/audio_samples/ to test with
    audio_dir = root / "data/data/raw/audio_samples"
    candidates = sorted(audio_dir.glob("**/*.wav"))
    sample_transcript = "This is a placeholder transcript for a quick manual test."

    if not candidates:
        print(f"No .wav files found under {audio_dir} — check the folder path.")
        sys.exit(0)

    sample_audio = candidates[0]
    print(f"Testing with: {sample_audio}")

    metrics = analyze_pace_and_pauses(str(sample_audio), sample_transcript)
    print(metrics)

Testing with: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\data\data\raw\audio_samples\Actor_01\03-01-01-01-01-01-01.wav
{'duration_sec': 3.303, 'word_count': 10, 'wpm': 181.64, 'pause_count': 0, 'total_pause_sec': 0.0, 'avg_pause_sec': 0.0, 'longest_pause_sec': 0.0, 'pause_ratio': 0.0}


In [19]:
"""
Person 3 — Part 4a: Fit and Save the Feature Scaler
======================================================
The original scaler used to produce train/val/test.csv (in notebook 02,
cell 11) was never saved. This refits an identical StandardScaler from
data/data/processed/audio/audio_features_raw.csv — verified to reproduce
train.csv's scaled values exactly (max abs diff ~1e-7, floating point noise).

Run this ONCE. It saves models/audio_feature_scaler.joblib, which
analyze_delivery() (Part 4b) loads to scale features from new audio files.
"""

import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler


def _find_project_root(marker="data/data/processed/audio/train.csv", max_up=4):
    p = Path.cwd()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f"Could not locate '{marker}' from {Path.cwd()}")


PROJECT_ROOT = _find_project_root()
RAW_FEATURES_PATH = PROJECT_ROOT / "data/data/processed/audio/audio_features_raw.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
SCALER_PATH = MODEL_DIR / "audio_feature_scaler.joblib"

FEATURE_COLS = [f"mfcc_{i}" for i in range(13)] + [
    "pitch_mean",
    "energy_mean",
    "zcr_mean",
    "duration",
]

raw_df = pd.read_csv(RAW_FEATURES_PATH)
print(f"Loaded raw features: {raw_df.shape}")

missing = [c for c in FEATURE_COLS if c not in raw_df.columns]
if missing:
    raise ValueError(f"audio_features_raw.csv missing expected columns: {missing}")

scaler = StandardScaler()
scaler.fit(raw_df[FEATURE_COLS])

joblib.dump(scaler, SCALER_PATH)
print(f"Saved scaler to: {SCALER_PATH}")
print("\nSanity check — scaler mean per feature:")
print(pd.Series(scaler.mean_, index=FEATURE_COLS))

Loaded raw features: (1440, 21)
Saved scaler to: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\models\audio_feature_scaler.joblib

Sanity check — scaler mean per feature:
mfcc_0         -394.453423
mfcc_1           72.994630
mfcc_2          -18.706284
mfcc_3           11.015809
mfcc_4          -17.253466
mfcc_5          -12.935085
mfcc_6          -22.058856
mfcc_7          -17.993505
mfcc_8          -12.462907
mfcc_9           -7.212847
mfcc_10         -11.996890
mfcc_11          -4.069906
mfcc_12          -8.883755
pitch_mean     1343.479993
energy_mean       0.022595
zcr_mean          0.143048
duration          1.732832
dtype: float64


In [20]:
"""
Person 3 — Part 4b: analyze_delivery(audio_file, transcript)
================================================================
Combines everything from Parts 1-3 into the single deliverable function:

    analyze_delivery(audio_file, transcript) -> confidence score + pace + pause metrics

Requires (run these once first, in order, if not already done):
  1. train_confidence_classifier.py   -> models/confidence_classifier.joblib
                                          models/confidence_label_encoder.joblib
  2. fit_feature_scaler.py            -> models/audio_feature_scaler.joblib

Feature extraction below is copied EXACTLY from notebook 02's
extract_audio_features() (TARGET_SR=16000, top_db=20 trim, n_mfcc=13,
piptrack pitch, RMS energy, ZCR) so it matches how train.csv was built.
"""

import librosa
import numpy as np
import pandas as pd
import joblib
from pathlib import Path


# ---------------------------------------------------------------------------
# 0. Locate project root + load saved artifacts
# ---------------------------------------------------------------------------
def _find_project_root(marker="data/data/processed/audio/train.csv", max_up=4):
    p = Path.cwd()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f"Could not locate '{marker}' from {Path.cwd()}")


PROJECT_ROOT = _find_project_root()
MODEL_DIR = PROJECT_ROOT / "models"

CLASSIFIER_PATH = MODEL_DIR / "confidence_classifier.joblib"
LABEL_ENCODER_PATH = MODEL_DIR / "confidence_label_encoder.joblib"
SCALER_PATH = MODEL_DIR / "audio_feature_scaler.joblib"

FEATURE_COLS = [f"mfcc_{i}" for i in range(13)] + [
    "pitch_mean",
    "energy_mean",
    "zcr_mean",
    "duration",
]

TARGET_SR = 16000  # must match notebook 02


def _load_artifacts():
    for path, label in [
        (CLASSIFIER_PATH, "classifier (run train_confidence_classifier.py)"),
        (LABEL_ENCODER_PATH, "label encoder (run train_confidence_classifier.py)"),
        (SCALER_PATH, "feature scaler (run fit_feature_scaler.py)"),
    ]:
        if not path.exists():
            raise FileNotFoundError(f"Missing {label} at {path}")
    clf = joblib.load(CLASSIFIER_PATH)
    label_encoder = joblib.load(LABEL_ENCODER_PATH)
    scaler = joblib.load(SCALER_PATH)
    return clf, label_encoder, scaler


_CLF, _LABEL_ENCODER, _SCALER = _load_artifacts()


# ---------------------------------------------------------------------------
# 1. Raw feature extraction — copied exactly from notebook 02
# ---------------------------------------------------------------------------
def extract_audio_features(file_path: str):
    """Same logic as notebook 02's extract_audio_features(). Returns a dict
    of RAW (unscaled) features, or None if the file can't be processed."""
    try:
        audio, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)
        audio_trimmed, _ = librosa.effects.trim(audio, top_db=20)

        if len(audio_trimmed) == 0:
            return None

        mfccs = librosa.feature.mfcc(y=audio_trimmed, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)

        pitches, magnitudes = librosa.piptrack(y=audio_trimmed, sr=sr)
        pitch_values = pitches[magnitudes > np.median(magnitudes)]
        pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0

        rms = librosa.feature.rms(y=audio_trimmed)
        energy_mean = np.mean(rms)

        zcr = librosa.feature.zero_crossing_rate(audio_trimmed)
        zcr_mean = np.mean(zcr)

        duration = len(audio_trimmed) / sr

        return {
            **{f"mfcc_{i}": mfccs_mean[i] for i in range(13)},
            "pitch_mean": pitch_mean,
            "energy_mean": energy_mean,
            "zcr_mean": zcr_mean,
            "duration": duration,
        }
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


# ---------------------------------------------------------------------------
# 2. Pace / pause metrics — from Part 3 (pace_pause_metrics.py)
#    NOTE: uses sr=None (original sample rate) here, matching Part 3 as
#    already validated — separate from the TARGET_SR=16000 load above,
#    which is only for the classifier's MFCC/pitch/energy/ZCR features.
# ---------------------------------------------------------------------------
def analyze_pace_and_pauses(
    audio_path: str,
    transcript: str,
    top_db: int = 30,
    min_pause_sec: float = 0.3,
):
    y, sr = librosa.load(audio_path, sr=None)
    duration_sec = librosa.get_duration(y=y, sr=sr)

    word_count = len(transcript.split())
    wpm = (word_count / (duration_sec / 60.0)) if duration_sec > 0 else 0.0

    intervals = librosa.effects.split(y, top_db=top_db)
    pause_durations = []
    if len(intervals) > 1:
        for i in range(len(intervals) - 1):
            gap_sec = (intervals[i + 1][0] - intervals[i][1]) / sr
            if gap_sec >= min_pause_sec:
                pause_durations.append(gap_sec)

    pause_count = len(pause_durations)
    total_pause_sec = float(sum(pause_durations))
    avg_pause_sec = float(np.mean(pause_durations)) if pause_durations else 0.0
    longest_pause_sec = float(max(pause_durations)) if pause_durations else 0.0
    pause_ratio = (total_pause_sec / duration_sec) if duration_sec > 0 else 0.0

    return {
        "duration_sec": round(duration_sec, 3),
        "word_count": word_count,
        "wpm": round(wpm, 2),
        "pause_count": pause_count,
        "total_pause_sec": round(total_pause_sec, 3),
        "avg_pause_sec": round(avg_pause_sec, 3),
        "longest_pause_sec": round(longest_pause_sec, 3),
        "pause_ratio": round(pause_ratio, 3),
    }


# ---------------------------------------------------------------------------
# 3. THE DELIVERABLE: analyze_delivery()
# ---------------------------------------------------------------------------
def analyze_delivery(audio_file: str, transcript: str):
    """
    Full delivery analysis for one audio file + its transcript.

    Parameters
    ----------
    audio_file : str
        Path to a .wav file.
    transcript : str
        The transcript text for this audio (caller supplies it — from
        Whisper output, a CSV column, wherever).

    Returns
    -------
    dict with:
        confidence_label   : predicted class ('confident' / 'nervous' / 'neutral')
        confidence_scores  : dict of class -> predicted probability
        pace_pause_metrics : dict from analyze_pace_and_pauses()
    """
    # --- Confidence classification ---
    raw_features = extract_audio_features(audio_file)
    if raw_features is None:
        raise ValueError(f"Could not extract features from {audio_file}")

    features_df = pd.DataFrame([raw_features])[FEATURE_COLS]
    features_scaled = pd.DataFrame(
        _SCALER.transform(features_df), columns=FEATURE_COLS
    )

    pred_idx = _CLF.predict(features_scaled)[0]
    pred_label = _LABEL_ENCODER.inverse_transform([pred_idx])[0]

    proba = _CLF.predict_proba(features_scaled)[0]
    confidence_scores = {
        cls: round(float(p), 4) for cls, p in zip(_LABEL_ENCODER.classes_, proba)
    }

    # --- Pace / pause metrics ---
    pace_pause = analyze_pace_and_pauses(audio_file, transcript)

    return {
        "confidence_label": pred_label,
        "confidence_scores": confidence_scores,
        "pace_pause_metrics": pace_pause,
    }


# ---------------------------------------------------------------------------
# Quick manual test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    audio_dir = PROJECT_ROOT / "data/data/raw/audio_samples"
    candidates = sorted(audio_dir.glob("**/*.wav"))
    if not candidates:
        print(f"No .wav files found under {audio_dir}")
    else:
        sample_audio = str(candidates[0])
        sample_transcript = "This is a placeholder transcript for a quick manual test."
        print(f"Testing with: {sample_audio}\n")
        result = analyze_delivery(sample_audio, sample_transcript)
        print(result)

Testing with: c:\Users\Janessa\OneDrive\Desktop\AgenticAi\AI-Powered-Voice-Based-Mock-Interview-Coach\data\data\raw\audio_samples\Actor_01\03-01-01-01-01-01-01.wav

{'confidence_label': 'confident', 'confidence_scores': {'confident': 0.8184, 'nervous': 0.1341, 'neutral': 0.0475}, 'pace_pause_metrics': {'duration_sec': 3.303, 'word_count': 10, 'wpm': 181.64, 'pause_count': 0, 'total_pause_sec': 0.0, 'avg_pause_sec': 0.0, 'longest_pause_sec': 0.0, 'pause_ratio': 0.0}}
